# Entregável 2 — Avaliação da Qualidade do Sinal (SQI)

**Disciplina:** Aquisição de Biossinais  
**Equipe:** José Lessa & Matheus Rocha  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL (PhysioNet)  

---

## Objetivo

Avaliar a qualidade dos sinais de ECG do dataset PTB-XL antes de qualquer processamento 
estatístico ou filtragem. Este notebook aplica métricas de **Signal Quality Index (SQI)** 
para identificar e classificar segmentos de boa e má qualidade, estabelecendo critérios 
de decisão para rejeição automática de registros problemáticos.

### Conteúdo exigido pelo pipeline:
- Métricas de SQI: SNR, Kurtosis, Skewness, Entropia espectral
- Identificação de artefatos
- Critérios de decisão adotados
- Tabela com valores de SQI
- Gráficos comparativos (segmentos bons × ruins)
- Marcação de segmentos rejeitados

## 1. Configuração e Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import wfdb
from pathlib import Path
from scipy import stats, signal as sp_signal
import warnings
warnings.filterwarnings('ignore')

# Configurações de plot
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Diretórios
DATA_DIR = Path('../../data/ptb-xl')
FIG_DIR = Path('../figuras')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Configuração concluída.')

## 2. Funções Auxiliares de Carregamento

In [ ]:
def carregar_ecg(ecg_id, df, data_dir, sampling_rate=500):
    """Carrega um registro de ECG do PTB-XL."""
    col = 'filename_hr' if sampling_rate == 500 else 'filename_lr'
    filename = df.loc[ecg_id, col]
    record = wfdb.rdrecord(str(data_dir / filename))
    return record


# Carregar metadados
df = pd.read_csv(DATA_DIR / 'ptbxl_database.csv', index_col='ecg_id')
print(f'Dataset carregado: {len(df)} registros')

## 3. Definição das Métricas de SQI

Implementação de quatro métricas fundamentais de qualidade de sinal, conforme 
exigido pelo pipeline da disciplina.

In [ ]:
def calcular_snr(sinal, fs):
    """
    Calcula a Razão Sinal-Ruído (SNR) em dB.
    
    Estratégia: estima o 'sinal' como a componente de baixa frequência (0.5-40 Hz,
    faixa principal do ECG) e o 'ruído' como a diferença entre o sinal original 
    e a componente filtrada.
    
    Parâmetros:
        sinal: array 1D com o sinal de ECG
        fs: frequência de amostragem (Hz)
    
    Retorna:
        snr_db: SNR em decibéis
    """
    # Filtro passa-banda 0.5-40 Hz para isolar componente de sinal
    nyq = fs / 2
    low = 0.5 / nyq
    high = min(40.0 / nyq, 0.99)  # Garantir que não excede Nyquist
    
    try:
        b, a = sp_signal.butter(4, [low, high], btype='band')
        sinal_filtrado = sp_signal.filtfilt(b, a, sinal)
        ruido = sinal - sinal_filtrado
        
        potencia_sinal = np.mean(sinal_filtrado ** 2)
        potencia_ruido = np.mean(ruido ** 2)
        
        if potencia_ruido < 1e-12:
            return 60.0  # SNR muito alto (sinal quase sem ruído)
        
        snr_db = 10 * np.log10(potencia_sinal / potencia_ruido)
        return snr_db
    except Exception:
        return np.nan


def calcular_kurtosis(sinal):
    """
    Calcula a Curtose (kurtosis) do sinal.
    
    A curtose mede o 'peso' das caudas da distribuição. Para um ECG:
    - Valor alto (>5): pode indicar artefatos pontuais/spikes
    - Valor próximo de 3: distribuição semelhante à normal
    - Valor muito baixo (<1): sinal possivelmente achatado/saturado
    
    Usa a convenção de excesso de curtose (Fisher), onde normal = 0.
    """
    return stats.kurtosis(sinal, fisher=True)


def calcular_skewness(sinal):
    """
    Calcula a Assimetria (skewness) do sinal.
    
    ECG bem adquirido tende a ter skewness próximo de zero.
    Valores muito diferentes de zero podem indicar:
    - Drift de baseline (deslocamento assimétrico)
    - Artefatos unidirecionais
    - Saturação em apenas um dos limites
    """
    return stats.skew(sinal)


def calcular_entropia_espectral(sinal, fs):
    """
    Calcula a Entropia Espectral do sinal.
    
    A entropia espectral mede a 'desordem' da distribuição de potência no
    domínio da frequência. Valores:
    - Próximo de 1: potência distribuída uniformemente (ruído branco)
    - Próximo de 0: potência concentrada em poucas frequências (sinal periódico)
    - ECG normal: valores intermediários (tem componentes bem definidas mas não é puro)
    
    Parâmetros:
        sinal: array 1D
        fs: frequência de amostragem
    
    Retorna:
        entropia: valor normalizado entre 0 e 1
    """
    # Calcular PSD via Welch
    freqs, psd = sp_signal.welch(sinal, fs=fs, nperseg=min(256, len(sinal)))
    
    # Normalizar PSD para obter distribuição de probabilidade
    psd_norm = psd / (np.sum(psd) + 1e-12)
    
    # Remover zeros para evitar log(0)
    psd_norm = psd_norm[psd_norm > 0]
    
    # Entropia de Shannon normalizada
    entropia = -np.sum(psd_norm * np.log2(psd_norm))
    entropia_max = np.log2(len(psd_norm))  # Entropia máxima possível
    
    return entropia / entropia_max if entropia_max > 0 else 0


def calcular_todas_metricas(sinal, fs):
    """
    Calcula todas as métricas de SQI para um sinal de um canal.
    
    Retorna dicionário com: SNR, kurtosis, skewness, entropia espectral.
    """
    return {
        'snr_db': calcular_snr(sinal, fs),
        'kurtosis': calcular_kurtosis(sinal),
        'skewness': calcular_skewness(sinal),
        'entropia_espectral': calcular_entropia_espectral(sinal, fs)
    }


print('Funções de SQI definidas.')

## 4. Cálculo das Métricas de SQI para Amostra do Dataset

Calculamos as métricas de SQI para uma amostra representativa do dataset, 
analisando todos os 12 canais de cada registro.

In [ ]:
# Calcular SQI para uma amostra representativa
N_AMOSTRA = 500  # Número de registros a analisar
np.random.seed(42)
ids_amostra = np.random.choice(df.index, size=N_AMOSTRA, replace=False)

resultados_sqi = []

for idx, ecg_id in enumerate(ids_amostra):
    if idx % 100 == 0:
        print(f'Processando {idx}/{N_AMOSTRA}...')
    
    try:
        record = carregar_ecg(ecg_id, df, DATA_DIR, sampling_rate=500)
        fs = record.fs
        
        for ch_idx, ch_name in enumerate(record.sig_name):
            sinal = record.p_signal[:, ch_idx]
            metricas = calcular_todas_metricas(sinal, fs)
            metricas['ecg_id'] = ecg_id
            metricas['canal'] = ch_name
            resultados_sqi.append(metricas)
    except Exception as e:
        print(f'  Erro no registro {ecg_id}: {e}')

df_sqi = pd.DataFrame(resultados_sqi)
print(f'\nConcluído! {len(df_sqi)} canais analisados ({df_sqi["ecg_id"].nunique()} registros).')

In [ ]:
# Tabela resumo das métricas de SQI
print('=' * 70)
print('TABELA DE MÉTRICAS DE SQI — ESTATÍSTICAS DESCRITIVAS')
print('=' * 70)
print(df_sqi[['snr_db', 'kurtosis', 'skewness', 'entropia_espectral']].describe().round(4))

print('\n')
print('=' * 70)
print('MÉTRICAS POR DERIVAÇÃO (MÉDIA)')
print('=' * 70)
tabela_por_canal = df_sqi.groupby('canal')[['snr_db', 'kurtosis', 'skewness', 'entropia_espectral']].mean().round(4)
print(tabela_por_canal)

## 5. Visualização das Distribuições de SQI

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metricas_info = [
    ('snr_db', 'SNR (dB)', 'Razão Sinal-Ruído', 'steelblue'),
    ('kurtosis', 'Curtose (excesso)', 'Curtose', '#e67e22'),
    ('skewness', 'Assimetria', 'Assimetria (Skewness)', '#2ecc71'),
    ('entropia_espectral', 'Entropia Espectral (normalizada)', 'Entropia Espectral', '#9b59b6')
]

for ax, (col, xlabel, titulo, cor) in zip(axes.flat, metricas_info):
    dados = df_sqi[col].dropna()
    
    # Remover outliers extremos para visualização
    q1, q3 = dados.quantile(0.01), dados.quantile(0.99)
    dados_vis = dados[(dados >= q1) & (dados <= q3)]
    
    ax.hist(dados_vis, bins=60, color=cor, edgecolor='white', alpha=0.8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Contagem')
    ax.set_title(f'Distribuição de {titulo}', fontweight='bold')
    ax.axvline(dados.median(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mediana: {dados.median():.3f}')
    ax.axvline(dados.mean(), color='darkred', linestyle=':', linewidth=1.5,
               label=f'Média: {dados.mean():.3f}')
    ax.legend(fontsize=9)

plt.suptitle('Distribuição das Métricas de SQI — Dataset PTB-XL (amostra)', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'distribuicao_metricas_sqi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

In [ ]:
# Boxplots das métricas por derivação
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for ax, (col, xlabel, titulo, cor) in zip(axes.flat, metricas_info):
    dados_pivot = df_sqi.pivot_table(values=col, index='ecg_id', columns='canal')
    
    # Ordenar derivações na ordem padrão do ECG
    ordem = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
    colunas_existentes = [c for c in ordem if c in dados_pivot.columns]
    dados_pivot = dados_pivot[colunas_existentes]
    
    bp = ax.boxplot([dados_pivot[c].dropna() for c in colunas_existentes],
                    labels=colunas_existentes, patch_artist=True,
                    showfliers=False)  # Esconder outliers para clareza
    
    for patch in bp['boxes']:
        patch.set_facecolor(cor)
        patch.set_alpha(0.6)
    
    ax.set_xlabel('Derivação')
    ax.set_ylabel(xlabel)
    ax.set_title(f'{titulo} por Derivação', fontweight='bold')

plt.suptitle('Métricas de SQI por Derivação de ECG', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'sqi_por_derivacao_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 6. Critérios de Decisão para Rejeição de Segmentos

Definimos critérios objetivos para classificar cada canal como **BOM** ou **RUIM** 
com base nas métricas de SQI. Os limiares são fundamentados na literatura de 
processamento de ECG e ajustados empiricamente com base nas distribuições observadas.

### Critérios adotados:

| Métrica | Critério de rejeição | Justificativa |
|---------|---------------------|---------------|
| **SNR** | < 5 dB | SNR muito baixo indica ruído dominante |
| **Curtose** | > 20 ou < -1 | Valores extremos indicam spikes ou saturação |
| **Skewness** | \|skew\| > 3 | Assimetria excessiva sugere drift ou artefato |
| **Entropia espectral** | > 0.95 ou < 0.2 | Muito alta = ruído branco; muito baixa = flat-line |

In [ ]:
# Definir critérios de rejeição
def classificar_qualidade(row):
    """
    Classifica a qualidade de um canal com base nos critérios de SQI.
    
    Retorna:
        'BOM': passa em todos os critérios
        'RUIM': falha em pelo menos um critério
        motivos: lista de motivos de rejeição
    """
    motivos = []
    
    if pd.notna(row['snr_db']) and row['snr_db'] < 5:
        motivos.append('SNR baixo')
    
    if pd.notna(row['kurtosis']) and (row['kurtosis'] > 20 or row['kurtosis'] < -1):
        motivos.append('Curtose anormal')
    
    if pd.notna(row['skewness']) and abs(row['skewness']) > 3:
        motivos.append('Skewness excessivo')
    
    if pd.notna(row['entropia_espectral']):
        if row['entropia_espectral'] > 0.95:
            motivos.append('Entropia alta (ruído)')
        elif row['entropia_espectral'] < 0.2:
            motivos.append('Entropia baixa (flat-line)')
    
    qualidade = 'RUIM' if motivos else 'BOM'
    return qualidade, motivos


# Aplicar classificação
classificacoes = df_sqi.apply(classificar_qualidade, axis=1)
df_sqi['qualidade'] = [c[0] for c in classificacoes]
df_sqi['motivos_rejeicao'] = ['; '.join(c[1]) if c[1] else '' for c in classificacoes]

print('=== RESULTADO DA CLASSIFICAÇÃO DE QUALIDADE ===')
print(f'\nTotal de canais analisados: {len(df_sqi)}')
print(f'\nDistribuição:')
print(df_sqi['qualidade'].value_counts())
print(f'\nPercentual de canais BOM: {(df_sqi["qualidade"] == "BOM").mean()*100:.1f}%')
print(f'Percentual de canais RUIM: {(df_sqi["qualidade"] == "RUIM").mean()*100:.1f}%')

In [ ]:
# Motivos de rejeição — quais são os mais frequentes?
motivos_todos = []
for m in df_sqi[df_sqi['qualidade'] == 'RUIM']['motivos_rejeicao']:
    motivos_todos.extend(m.split('; '))

if motivos_todos:
    motivos_contagem = pd.Series(motivos_todos).value_counts()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    cores = ['#e74c3c', '#e67e22', '#f1c40f', '#9b59b6', '#3498db']
    motivos_contagem.plot(kind='barh', color=cores[:len(motivos_contagem)], 
                          edgecolor='white', ax=ax)
    ax.set_xlabel('Número de canais rejeitados')
    ax.set_title('Motivos de Rejeição de Canais', fontweight='bold')
    
    # Adicionar valores nas barras
    for i, v in enumerate(motivos_contagem.values):
        ax.text(v + 1, i, str(v), va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'motivos_rejeicao.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura salva.')
else:
    print('Nenhum canal foi rejeitado!')

In [ ]:
# Análise por registro: quantos canais ruins em cada registro?
canais_ruins_por_registro = df_sqi[df_sqi['qualidade'] == 'RUIM'].groupby('ecg_id').size()
total_registros = df_sqi['ecg_id'].nunique()

# Registros completamente bons vs com pelo menos 1 canal ruim
registros_com_problema = canais_ruins_por_registro.index
registros_bons = set(df_sqi['ecg_id'].unique()) - set(registros_com_problema)

print('=== ANÁLISE POR REGISTRO ===')
print(f'Registros analisados: {total_registros}')
print(f'Registros totalmente limpos (todos 12 canais BOM): {len(registros_bons)} ({len(registros_bons)/total_registros*100:.1f}%)')
print(f'Registros com ≥1 canal RUIM: {len(registros_com_problema)} ({len(registros_com_problema)/total_registros*100:.1f}%)')

if len(canais_ruins_por_registro) > 0:
    print(f'\nDistribuição de canais ruins por registro problemático:')
    print(canais_ruins_por_registro.value_counts().sort_index())

## 7. Gráficos Comparativos: Segmentos Bons × Ruins

In [ ]:
def plotar_comparacao_bom_ruim(df_sqi, df_meta, data_dir, fig_dir):
    """
    Plota comparação lado a lado de um segmento BOM e um RUIM.
    """
    # Encontrar um exemplo BOM (alto SNR)
    bons = df_sqi[df_sqi['qualidade'] == 'BOM'].sort_values('snr_db', ascending=False)
    
    # Encontrar um exemplo RUIM (se existir)
    ruins = df_sqi[df_sqi['qualidade'] == 'RUIM'].sort_values('snr_db', ascending=True)
    
    if len(ruins) == 0:
        print('Nenhum segmento classificado como RUIM. Usando os piores como comparação.')
        ruins = df_sqi.sort_values('snr_db', ascending=True)
    
    # Pegar o melhor BOM e o pior RUIM
    melhor = bons.iloc[0]
    pior = ruins.iloc[0]
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    
    for col_idx, (registro, label, cor_sinal) in enumerate([
        (melhor, 'BOM (alto SNR)', '#2ecc71'),
        (pior, 'RUIM (baixo SNR)', '#e74c3c')
    ]):
        ecg_id = int(registro['ecg_id'])
        canal = registro['canal']
        record = carregar_ecg(ecg_id, df_meta, data_dir, sampling_rate=500)
        ch_idx = record.sig_name.index(canal)
        sinal = record.p_signal[:, ch_idx]
        fs = record.fs
        t = np.arange(len(sinal)) / fs
        
        # Sinal no tempo
        axes[0, col_idx].plot(t, sinal, color=cor_sinal, linewidth=0.7)
        axes[0, col_idx].set_xlabel('Tempo (s)')
        axes[0, col_idx].set_ylabel('Amplitude (mV)')
        axes[0, col_idx].set_title(
            f'Segmento {label}\n'
            f'Registro #{ecg_id}, Derivação {canal}\n'
            f'SNR={registro["snr_db"]:.1f} dB | Kurt={registro["kurtosis"]:.2f} | '
            f'Skew={registro["skewness"]:.2f} | Ent={registro["entropia_espectral"]:.3f}',
            fontweight='bold', fontsize=10
        )
        
        # Espectro
        freqs_psd, psd = sp_signal.welch(sinal, fs=fs, nperseg=min(512, len(sinal)))
        axes[1, col_idx].semilogy(freqs_psd, psd, color=cor_sinal, linewidth=1)
        axes[1, col_idx].set_xlabel('Frequência (Hz)')
        axes[1, col_idx].set_ylabel('PSD (mV²/Hz)')
        axes[1, col_idx].set_title(f'Espectro de Potência — {label}', fontweight='bold')
        axes[1, col_idx].set_xlim([0, 100])
        axes[1, col_idx].axvline(50, color='orange', linestyle='--', alpha=0.6, label='50 Hz')
        axes[1, col_idx].legend()
    
    plt.suptitle('Comparação: Segmentos BOM vs RUIM', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(fig_dir / 'comparacao_bom_ruim.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura salva.')


plotar_comparacao_bom_ruim(df_sqi, df, DATA_DIR, FIG_DIR)

In [ ]:
# Scatter plot: SNR vs Entropia Espectral colorido por qualidade
fig, ax = plt.subplots(figsize=(12, 8))

bons = df_sqi[df_sqi['qualidade'] == 'BOM']
ruins = df_sqi[df_sqi['qualidade'] == 'RUIM']

ax.scatter(bons['snr_db'], bons['entropia_espectral'], 
           c='#2ecc71', alpha=0.3, s=10, label=f'BOM ({len(bons)})')
if len(ruins) > 0:
    ax.scatter(ruins['snr_db'], ruins['entropia_espectral'], 
               c='#e74c3c', alpha=0.5, s=20, label=f'RUIM ({len(ruins)})', marker='x')

# Marcar limiares
ax.axvline(5, color='red', linestyle='--', alpha=0.5, label='Limiar SNR = 5 dB')
ax.axhline(0.95, color='orange', linestyle='--', alpha=0.5, label='Limiar Entropia = 0.95')
ax.axhline(0.2, color='orange', linestyle=':', alpha=0.5, label='Limiar Entropia = 0.2')

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('Entropia Espectral', fontsize=12)
ax.set_title('Mapa de Qualidade: SNR vs Entropia Espectral', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'mapa_qualidade_snr_entropia.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 8. Marcação de Segmentos Rejeitados

Geramos a lista final de segmentos (registros × canais) rejeitados para 
exclusão ou tratamento nos entregáveis subsequentes.

In [ ]:
# Lista de segmentos rejeitados
rejeitados = df_sqi[df_sqi['qualidade'] == 'RUIM'][['ecg_id', 'canal', 'snr_db', 'kurtosis', 
                                                      'skewness', 'entropia_espectral', 
                                                      'motivos_rejeicao']].copy()

print(f'Total de segmentos rejeitados: {len(rejeitados)}')
print(f'\nPrimeiros 20 segmentos rejeitados:')
if len(rejeitados) > 0:
    display(rejeitados.head(20).round(4))
else:
    print('Nenhum segmento rejeitado pelos critérios adotados.')
    print('\nOs 10 piores canais (por SNR) para referência:')
    piores = df_sqi.nsmallest(10, 'snr_db')[['ecg_id', 'canal', 'snr_db', 'kurtosis', 
                                               'skewness', 'entropia_espectral']]
    display(piores.round(4))

In [ ]:
# Visualizar um registro rejeitado com marcação visual dos canais ruins
if len(rejeitados) > 0:
    # Pegar o registro com mais canais rejeitados
    registro_pior = rejeitados.groupby('ecg_id').size().idxmax()
    canais_ruins = set(rejeitados[rejeitados['ecg_id'] == registro_pior]['canal'])
    
    record = carregar_ecg(registro_pior, df, DATA_DIR, sampling_rate=500)
    signal = record.p_signal
    fs = record.fs
    t = np.arange(signal.shape[0]) / fs
    channels = record.sig_name
    
    fig, axes = plt.subplots(6, 2, figsize=(18, 16), sharex=True)
    fig.suptitle(f'Registro #{registro_pior} — Canais rejeitados marcados em vermelho', 
                 fontsize=14, fontweight='bold', y=1.01)
    
    for i, ax in enumerate(axes.flat):
        if i < len(channels):
            eh_ruim = channels[i] in canais_ruins
            cor = '#e74c3c' if eh_ruim else '#2c3e50'
            bg = '#fff5f5' if eh_ruim else '#f8f9fa'
            
            ax.plot(t, signal[:, i], color=cor, linewidth=0.7)
            ax.set_facecolor(bg)
            
            status = '❌ REJEITADO' if eh_ruim else '✅ OK'
            ax.set_title(f'{channels[i]} — {status}', fontsize=10, fontweight='bold', 
                         loc='left', color=cor)
            ax.set_ylabel('mV')
        else:
            ax.set_visible(False)
    
    axes[-1, 0].set_xlabel('Tempo (s)')
    axes[-1, 1].set_xlabel('Tempo (s)')
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'registro_canais_rejeitados.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Registro #{registro_pior}: {len(canais_ruins)} canais rejeitados: {canais_ruins}')
else:
    print('Nenhum registro com canais rejeitados para visualizar.')
    print('O dataset PTB-XL apresenta qualidade geral elevada.')

In [ ]:
# Salvar resultados de SQI em CSV para referência futura
output_path = FIG_DIR.parent / 'sqi_resultados.csv'
df_sqi.to_csv(output_path, index=False)
print(f'Resultados de SQI salvos em: {output_path}')

## 9. Tabela Final de SQI

Tabela consolidada com as métricas de SQI e decisão de qualidade.

In [ ]:
# Tabela resumo final
print('=' * 80)
print('TABELA FINAL DE SQI — RESUMO DA AVALIAÇÃO DE QUALIDADE')
print('=' * 80)

resumo = {
    'Métrica': ['SNR (dB)', 'Curtose', 'Skewness', 'Entropia Espectral'],
    'Média': [
        df_sqi['snr_db'].mean(),
        df_sqi['kurtosis'].mean(),
        df_sqi['skewness'].mean(),
        df_sqi['entropia_espectral'].mean()
    ],
    'Mediana': [
        df_sqi['snr_db'].median(),
        df_sqi['kurtosis'].median(),
        df_sqi['skewness'].median(),
        df_sqi['entropia_espectral'].median()
    ],
    'Desvio Padrão': [
        df_sqi['snr_db'].std(),
        df_sqi['kurtosis'].std(),
        df_sqi['skewness'].std(),
        df_sqi['entropia_espectral'].std()
    ],
    'Limiar de Rejeição': ['< 5 dB', '> 20 ou < -1', '|skew| > 3', '> 0.95 ou < 0.2'],
    '% Rejeitados': [
        f"{(df_sqi['snr_db'] < 5).mean()*100:.1f}%",
        f"{((df_sqi['kurtosis'] > 20) | (df_sqi['kurtosis'] < -1)).mean()*100:.1f}%",
        f"{(df_sqi['skewness'].abs() > 3).mean()*100:.1f}%",
        f"{((df_sqi['entropia_espectral'] > 0.95) | (df_sqi['entropia_espectral'] < 0.2)).mean()*100:.1f}%"
    ]
}

df_resumo = pd.DataFrame(resumo)
display(df_resumo.round(4))

print(f'\n--- RESULTADO FINAL ---')
print(f'Total de registros analisados: {df_sqi["ecg_id"].nunique()}')
print(f'Total de canais analisados: {len(df_sqi)}')
print(f'Canais classificados como BOM: {(df_sqi["qualidade"] == "BOM").sum()} ({(df_sqi["qualidade"] == "BOM").mean()*100:.1f}%)')
print(f'Canais classificados como RUIM: {(df_sqi["qualidade"] == "RUIM").sum()} ({(df_sqi["qualidade"] == "RUIM").mean()*100:.1f}%)')

## 10. Conclusões e Interpretação

### Resumo da Avaliação de Qualidade

A avaliação de SQI aplicada a uma amostra representativa do dataset PTB-XL revelou 
a qualidade geral dos registros de ECG. Os principais achados são:

1. **SNR:** A maioria dos registros apresenta SNR adequado para análise, com valores 
   tipicamente acima do limiar de 5 dB

2. **Curtose:** Os valores observados estão dentro da faixa esperada para sinais de ECG, 
   indicando ausência generalizada de spikes ou saturação

3. **Skewness:** A maioria dos canais apresenta assimetria próxima de zero, consistente 
   com sinais de ECG bem adquiridos

4. **Entropia Espectral:** Valores intermediários predominam, indicando que os sinais 
   possuem conteúdo espectral estruturado (não são ruído branco)

### Critérios de Decisão

Os limiares adotados (SNR < 5 dB, |kurtosis| > 20, |skewness| > 3, entropia fora de 
0.2–0.95) permitem identificar automaticamente os segmentos problemáticos. Os segmentos 
marcados como RUIM serão tratados nos entregáveis subsequentes:
- **Entregável 4:** Limpeza e filtragem dos segmentos recuperáveis
- **Entregável 5:** Segmentação excluindo trechos irrecuperáveis

### Limitações

- Os limiares foram definidos empiricamente e podem precisar de ajuste fino
- A análise foi feita no sinal completo de 10s, sem segmentação intra-registro
- O PTB-XL, por ser de origem hospitalar, tende a ter qualidade superior a 
  bases coletadas com dispositivos vestíveis